# Lesson 4: Persistence and Streaming

In [ ]:
import subprocess
import sys
from pathlib import Path

print("Kernel Python:", sys.executable)

_here = Path.cwd()
_req = _here / "requirements-lesson-04.txt"
if not _req.exists():
    _req = _here.parent / "requirements-lesson-04.txt"

if _req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Installed from", _req)
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "langgraph", "langgraph-checkpoint-sqlite", "langchain-core", "langchain-openai",
        "langchain-community", "tavily-python", "python-dotenv", "httpx", "aiosqlite", "ipykernel",
    ])
    print("Installed packages via pip (requirements file not found).")

Kernel Python: /opt/anaconda3/bin/python
Installed from /Users/mkenne16/Documents/LangGraph/requirements-lesson-04.txt


## Agent with tools and checkpointing

Same ReAct-style pattern as earlier lessons: LLM node, tool node, conditional edge on `tool_calls`.

In [2]:
from dotenv import load_dotenv

_ = load_dotenv()

import operator
from typing import Annotated, TypedDict

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

tool = TavilySearchResults(max_results=2)


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def exists_action(self, state: AgentState):
        result = state["messages"][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        print("Back to the model!")
        return {"messages": results}


prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOpenAI(model="gpt-4o")

/var/folders/2p/x0g54f5s6_v5t47f_25hgyww0000gq/T/ipykernel_24231/1889474114.py:13: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=2)


## Persistence: `thread_id` and `SqliteSaver`

Checkpoints store graph state per **`thread_id`**. Reusing `thread_id="1"` lets the model see prior turns. A new `thread_id` starts a fresh conversation.

Use **`with SqliteSaver.from_conn_string(...)`** so the DB connection stays open for the whole demo (`:memory:` is wiped when the connection closes).

In [3]:
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string(":memory:") as memory:
    abot = Agent(model, [tool], system=prompt, checkpointer=memory)

    messages = [HumanMessage(content="What is the weather in sf?")]
    thread = {"configurable": {"thread_id": "1"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v["messages"])

    messages = [HumanMessage(content="What about in la?")]
    thread = {"configurable": {"thread_id": "1"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

    messages = [HumanMessage(content="Which one is warmer?")]
    thread = {"configurable": {"thread_id": "1"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

    messages = [HumanMessage(content="Which one is warmer?")]
    thread = {"configurable": {"thread_id": "2"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 151, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_899261a2ad', 'id': 'chatcmpl-DN6bGQ0SU63nJrqaaFk9sLotzksGY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2270-6687-7760-a3ad-f54c92897ad2-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_kTMRVgJKJ4SaESRCIO9iGpRA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 22, 'total_tokens': 173, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'a

## Streaming tokens (`astream_events`)

Use **`AsyncSqliteSaver`** with **`async with`** and iterate **`astream_events`** with `version="v1"`. `on_chat_model_stream` yields chunks; empty `content` often means the model is choosing a tool instead of emitting text.

In [4]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver


async def stream_tokens():
    async with AsyncSqliteSaver.from_conn_string(":memory:") as memory:
        abot = Agent(model, [tool], system=prompt, checkpointer=memory)
        messages = [HumanMessage(content="What is the weather in SF?")]
        thread = {"configurable": {"thread_id": "4"}}
        async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
            kind = event["event"]
            if kind == "on_chat_model_stream":
                content = event["data"]["chunk"].content
                if content:
                    # Empty content often means the model is asking for a tool.
                    print(content, end="|")
    print()


await stream_tokens()

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco weather October 12 2023'}, 'id': 'call_Ifc5t5D6tlx2M7I1lZQIRZUq', 'type': 'tool_call'}
Back to the model!
The| current| weather| in| San| Francisco| is| around| |68|°F| and| sunny|.| The| weather| typically| involves| varying| temperatures| with| highs| around| |70|°F| and| lows| around| |57|°F| in| October|.| The| wind| is| typically| from| the| west| at| approximately| |14|.|5| mph|,| with| humidity| around| |50|%.|
